In [1]:
import os
import json
import pandas as pd
import numpy as np
import torch
import faiss
import ollama

from sentence_transformers import SentenceTransformer

print("CUDA:", torch.cuda.is_available())

CUDA: True


In [2]:
df = pd.read_csv(
    "../data/processed/m_absa_selected.csv"
)

reviews = (
    df[["text", "language", "domain"]]
    .drop_duplicates(subset=["text"])
    .reset_index(drop=True)
)

train_reviews = (
    reviews[
        reviews["text"].isin(
            df[df["split"] == "train"]["text"].unique()
        )
    ]
    .reset_index(drop=True)
)

index = faiss.read_index(
    "../vector_store/index.faiss"
)

embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("Reviews:", len(reviews))
print("Train reviews:", len(train_reviews))
print("FAISS vectors:", index.ntotal)
print("Embedding device:", embedding_model.device)

Reviews: 17795
Train reviews: 10355
FAISS vectors: 10355
Embedding device: cuda:0


In [3]:
def retrieve_reviews(query, top_k=5, domain=None):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    search_k = max(top_k * 5, 20)

    scores, indices = index.search(
        query_embedding,
        search_k
    )

    results = train_reviews.iloc[indices[0]].copy()
    results["score"] = scores[0]

    if domain is not None:
        results = results[
            results["domain"] == domain
        ]

    return (
        results
        .sort_values("score", ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )

In [4]:
def build_context(results):

    parts = []

    for i, row in results.iterrows():

        parts.append(
            f"[Review {i + 1}]\n"
            f"Language: {row['language']}\n"
            f"Domain: {row['domain']}\n"
            f"Similarity: {row['score']:.3f}\n"
            f"Text: {row['text']}"
        )

    return "\n\n".join(parts)

In [5]:
def generate_llm_only(query):

    prompt = f"""
You are a review intelligence assistant.

Analyze the following customer-review query.

Query:
{query}

Return ONLY valid JSON:

{{
    "main_finding": "...",
    "common_opinion": "...",
    "negative_signals": [],
    "key_aspects": [],
    "conclusion": "..."
}}

Rules:
- Answer based only on your general language understanding.
- You do NOT have access to the review database.
- Do NOT invent specific customer evidence.
- Keep the answer concise.
"""

    response = ollama.chat(
        model="qwen3:8b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.message.content

In [6]:
def generate_rag(query, domain, top_k=5):

    results = retrieve_reviews(
        query=query,
        top_k=top_k,
        domain=domain
    )

    context = build_context(results)

    prompt = f"""
You are a review intelligence assistant.

Analyze the query using ONLY the retrieved customer reviews.

Query:
{query}

Retrieved reviews:
{context}

Return ONLY valid JSON:

{{
    "main_finding": "...",
    "common_opinion": "...",
    "negative_signals": [],
    "key_aspects": [],
    "evidence": [],
    "conclusion": "..."
}}

Rules:
- Use only the retrieved reviews as evidence.
- Do not invent customer opinions.
- Do not make claims unsupported by the retrieved reviews.
- If there are no negative signals, return [].
- Include 2-3 short evidence statements.
"""

    response = ollama.chat(
        model="qwen3:8b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.message.content, results

In [7]:
evaluation_queries = [
    {
        "id": "hotel_en",
        "query": "The hotel room was very clean",
        "language": "en",
        "domain": "hotel"
    },
    {
        "id": "hotel_ru",
        "query": "Номер в отеле был очень чистым",
        "language": "ru",
        "domain": "hotel"
    },
    {
        "id": "hotel_de",
        "query": "Das Hotelzimmer war sehr sauber",
        "language": "de",
        "domain": "hotel"
    },
    {
        "id": "laptop_en",
        "query": "The laptop battery drains very quickly",
        "language": "en",
        "domain": "laptop"
    },
    {
        "id": "laptop_ru",
        "query": "Батарея ноутбука быстро разряжается",
        "language": "ru",
        "domain": "laptop"
    },
    {
        "id": "laptop_de",
        "query": "Der Akku des Laptops hält nicht lange",
        "language": "de",
        "domain": "laptop"
    },
    {
        "id": "restaurant_en",
        "query": "The food was delicious but the service was slow",
        "language": "en",
        "domain": "restaurant"
    },
    {
        "id": "restaurant_ru",
        "query": "Еда была вкусной, но обслуживание было медленным",
        "language": "ru",
        "domain": "restaurant"
    },
    {
        "id": "restaurant_de",
        "query": "Das Essen war lecker, aber der Service war langsam",
        "language": "de",
        "domain": "restaurant"
    }
]

In [ ]:
evaluation_results = []

for item in evaluation_queries:

    print("=" * 100)
    print(item["id"])
    print("Language:", item["language"])
    print("Domain:", item["domain"])
    print("Query:", item["query"])

    # ==========================================================
    # 1. LLM ONLY
    # ==========================================================

    print("\n[1] LLM ONLY")

    llm_answer = generate_llm_only(
        item["query"]
    )

    print(llm_answer)

    # ==========================================================
    # 2. RAG
    # ==========================================================

    print("\n[2] RAG")

    rag_answer, retrieved = generate_rag(
        query=item["query"],
        domain=item["domain"],
        top_k=5
    )

    print(rag_answer)

    # ==========================================================
    # 3. Save results
    # ==========================================================

    evaluation_results.append({
        "id": item["id"],
        "query": item["query"],
        "language": item["language"],
        "domain": item["domain"],
        "llm_only": llm_answer,
        "rag": rag_answer,
        "top_similarity": retrieved["score"].iloc[0],
        "mean_similarity": retrieved["score"].mean()
    })

    print("\n")

{
    "main_finding": "Rooms were clean",
    "common_opinion": "Positive",
    "negative_signals": [],
    "key_aspects": ["Room cleanliness", "Staff friendliness"],
    "evidence": [
        "Отель имеет много особенностей, а номера были чистыми.",
        "Hotel hat viel Charakter und die Zimmer waren sauber.",
        "The rooms were clean hotel staff friendly ."
    ],
    "conclusion": "The reviews confirm that the hotel rooms were clean, aligning with the query's statement."
}


hotel_de
Language: de
Domain: hotel
Query: Das Hotelzimmer war sehr sauber

[1] LLM ONLY


In [ ]:
evaluation_df = pd.DataFrame(
    evaluation_results
)

evaluation_df.to_csv(
    "../data/processed/rag_llm_evaluation.csv",
    index=False,
    encoding="utf-8"
)

evaluation_df

In [ ]:
retrieval_stats = evaluation_df[
    [
        "language",
        "domain",
        "top_similarity",
        "mean_similarity"
    ]
].copy()

print(
    retrieval_stats.groupby("language")[
        ["top_similarity", "mean_similarity"]
    ].mean()
)

print()

print(
    retrieval_stats.groupby("domain")[
        ["top_similarity", "mean_similarity"]
    ].mean()
)

In [ ]:
from IPython.display import display


def interactive_demo(query, domain=None, top_k=5):

    print("=" * 100)
    print("REVIEWLENS AI — INTERACTIVE DEMO")
    print("=" * 100)

    print("\nUSER QUERY:")
    print(query)

    # ==========================================================
    # 1. LLM ONLY
    # ==========================================================

    print("\n" + "=" * 100)
    print("1. LLM ONLY — WITHOUT RETRIEVED CONTEXT")
    print("=" * 100)

    llm_answer = generate_llm_only(query)

    print(llm_answer)

    # ==========================================================
    # 2. RAG RETRIEVAL
    # ==========================================================

    print("\n" + "=" * 100)
    print("2. FAISS RETRIEVAL")
    print("=" * 100)

    rag_answer, retrieved = generate_rag(
        query=query,
        domain=domain,
        top_k=top_k
    )

    display(
        retrieved[
            ["text", "language", "domain", "score"]
        ].style.format(
            {"score": "{:.3f}"}
        )
    )

    # ==========================================================
    # 3. CONTEXT
    # ==========================================================

    context = build_context(retrieved)

    print("\n" + "=" * 100)
    print("3. CONTEXT SENT TO QWEN")
    print("=" * 100)

    print(context)

    # ==========================================================
    # 4. RAG ANSWER
    # ==========================================================

    print("\n" + "=" * 100)
    print("4. RAG + QWEN")
    print("=" * 100)

    print(rag_answer)

    # ==========================================================
    # 5. PIPELINE
    # ==========================================================

    print("\n" + "=" * 100)
    print("PIPELINE")
    print("=" * 100)

    print("""
LLM ONLY:
Query → Qwen → Answer

RAG:
Query → Multilingual → Embedding → FAISS → Top-K Reviews → Context → Qwen → Evidence-based → Answer
""")

    return {
        "query": query,
        "llm_only": llm_answer,
        "retrieved_reviews": retrieved,
        "context": context,
        "rag_answer": rag_answer
    }

In [ ]:
demo = interactive_demo(
    query="The restaurant food was good but the service was slow",
    domain="restaurant",
    top_k=5
)

In [ ]:
demo = interactive_demo(
    query="Батарея ноутбука быстро разряжается",
    domain="laptop",
    top_k=5
)

In [ ]:
demo = interactive_demo(
    query="Das Hotelzimmer war sauber, aber das Personal war unfreundlich",
    domain="hotel",
    top_k=5
)